In [ ]:
import pandas as pd
import numpy as np

FILE = "Copy of Master Data_290102026 2 - Copy.xlsx"

df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()

# Clean Category
df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)


In [ ]:
df = df[df["Category"].isin(["repeater", "stranger"])].copy()


In [ ]:
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]
    

In [ ]:
T120_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

grp = df.groupby("Child Part", sort=False)

parts = grp.agg({
    "effective_daily_demand": "sum",
    "Inventory_25": "first",
    "Minimum Quantity": "first",
    "Cycle Time": "first",
    "Vertical Machines": lambda x: [
        m.strip()
        for m in ",".join(x.dropna().astype(str)).split(",")
        if m.strip() in T120_MACHINES
    ],
    "Category": "first"
}).reset_index()


In [ ]:
parts = parts[parts["Vertical Machines"].map(len) > 0].copy()


In [ ]:
parts.rename(columns={
    "Child Part": "part",
    "effective_daily_demand": "daily_demand",
    "Inventory_25": "inventory",
    "Minimum Quantity": "min_qty",
    "Cycle Time": "cycle_time",
    "Vertical Machines": "machines"
}, inplace=True)


In [ ]:
parts["net_required_qty"] = (
    parts["daily_demand"]
    + parts["min_qty"]
    - parts["inventory"]
).clip(lower=0)


In [ ]:
rows = []

for _, r in parts.iterrows():
    for m in r["machines"]:
        rows.append({
            "part": r["part"],
            "category": r["Category"],
            "machine": m,
            "daily_demand": r["daily_demand"],
            "inventory": r["inventory"],
            "cycle_time": r["cycle_time"],
            "net_required_qty": r["net_required_qty"]
        })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("No Repeater/Stranger parts found for 120T machines")


In [ ]:
rows = []

for _, r in parts.iterrows():
    for m in r["machines"]:
        rows.append({
            "part": r["part"],
            "category": r["Category"],
            "machine": m,
            "daily_demand": r["daily_demand"],
            "inventory": r["inventory"],
            "cycle_time": r["cycle_time"],
            "net_required_qty": r["net_required_qty"]
        })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("No Repeater/Stranger parts found for 120T machines")


In [ ]:
CHANGEOVER = 40
CAPACITY = 1320
TARGET_DAYS = 3

def compute_score(row):
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)

    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    prod_time = qty_if_made * row["cycle_time"]

    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)


In [ ]:
dfm["score"] = dfm.apply(compute_score, axis=1)


In [ ]:
selected = []

for m, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(3))

selected = pd.concat(selected).reset_index(drop=True)


In [ ]:
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]


In [ ]:
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== DAILY PLAN (120T MACHINES ONLY) =====")
print(final_plan)


In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# CONFIG
# =========================================================
FILE = "Copy of Master Data_290102026 2 - Copy.xlsx"

T120_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

CHANGEOVER = 40          # minutes
CAPACITY = 1320          # minutes per machine per day
TARGET_DAYS = 3
MAX_PARTS_PER_MACHINE = 3

# =========================================================
# STEP 1: LOAD & CLEAN DATA
# =========================================================
df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()

df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Only Repeater & Stranger
df = df[df["Category"].isin(["repeater", "stranger"])].copy()

# =========================================================
# STEP 2: EFFECTIVE DAILY DEMAND
# =========================================================
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]

# =========================================================
# STEP 3: PART-LEVEL AGGREGATION (NO MACHINES HERE)
# =========================================================
part_level = (
    df.groupby("Child Part", sort=False)
      .agg({
          "effective_daily_demand": "sum",
          "Inventory_25": "first",
          "Minimum Quantity": "first",
          "Cycle Time": "first",
          "Category": "first"
      })
      .reset_index()
)

# =========================================================
# STEP 4: NET REQUIRED QTY (YOUR LOGIC)
# =========================================================
part_level["net_required_qty"] = (
    part_level["effective_daily_demand"]
    + part_level["Minimum Quantity"]
    - part_level["Inventory_25"]
).clip(lower=0)

# =========================================================
# STEP 5: BUILD PART–MACHINE TABLE (REAL DATA GRAIN)
# =========================================================
rows = []

for _, r in df.iterrows():
    machine = str(r["Vertical Machines"]).strip()

    if machine not in T120_MACHINES:
        continue

    p = part_level.loc[
        part_level["Child Part"] == r["Child Part"]
    ].iloc[0]

    rows.append({
        "part": r["Child Part"],
        "category": r["Category"],
        "machine": machine,
        "daily_demand": p["effective_daily_demand"],
        "inventory": p["Inventory_25"],
        "cycle_time": p["Cycle Time"],
        "net_required_qty": p["net_required_qty"]
    })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("❌ No Repeater/Stranger parts found on 120T machines")

print("✅ dfm created:", dfm.shape)

# =========================================================
# STEP 6: SCORE FUNCTION (HUMAN LOGIC)
# =========================================================
def compute_score(row):
    # Inventory pain
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    # Relief potential
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)

    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    # Production time
    prod_time = qty_if_made * row["cycle_time"]

    # Setup penalty
    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    # Monopoly penalty
    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)

dfm["score"] = dfm.apply(compute_score, axis=1)

# =========================================================
# STEP 7: SELECT TOP 2–3 PARTS PER MACHINE
# =========================================================
selected = []

for machine, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(MAX_PARTS_PER_MACHINE))

selected = pd.concat(selected).reset_index(drop=True)

# =========================================================
# STEP 8: QUANTITY TO PRODUCE (3-DAY INVENTORY)
# =========================================================
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]

# =========================================================
# FINAL OUTPUT
# =========================================================
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== FINAL DAILY PLAN (120T MACHINES ONLY) =====")
print(final_plan)


In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# CONFIG
# =========================================================
FILE = "Copy of Master Data_290102026 2 - Copy.xlsx"

# Human-known 120T machines
RAW_120T_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

# Normalize machine names → MP01, MP05, etc.
T120_MACHINES = {m.replace("-", "").upper() for m in RAW_120T_MACHINES}

CHANGEOVER = 40          # minutes
CAPACITY = 1320          # minutes/day
TARGET_DAYS = 3
MAX_PARTS_PER_MACHINE = 3

# =========================================================
# STEP 1: LOAD DATA
# =========================================================
df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()

# =========================================================
# STEP 2: CLEAN CATEGORY
# =========================================================
df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Keep only Repeater & Stranger
df = df[df["Category"].isin(["repeater", "stranger"])].copy()

# =========================================================
# STEP 3: CLEAN MACHINE NAMES (CRITICAL FIX)
# =========================================================
df["machine_clean"] = (
    df["Vertical Machines"]
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", "", regex=True)   # remove spaces/newlines
    .str.replace("-", "", regex=False)     # remove hyphens
)

print("DEBUG → Unique cleaned machines:")
print(sorted(df["machine_clean"].unique()))

# =========================================================
# STEP 4: EFFECTIVE DAILY DEMAND
# =========================================================
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]

# =========================================================
# STEP 5: PART-LEVEL AGGREGATION (NO MACHINES)
# =========================================================
part_level = (
    df.groupby("Child Part", sort=False)
      .agg({
          "effective_daily_demand": "sum",
          "Inventory_25": "first",
          "Minimum Quantity": "first",
          "Cycle Time": "first",
          "Category": "first"
      })
      .reset_index()
)

# =========================================================
# STEP 6: NET REQUIRED QTY (YOUR LOGIC)
# =========================================================
part_level["net_required_qty"] = (
    part_level["effective_daily_demand"]
    + part_level["Minimum Quantity"]
    - part_level["Inventory_25"]
).clip(lower=0)

# =========================================================
# STEP 7: BUILD PART–MACHINE TABLE (ROW-LEVEL, FIXED)
# =========================================================
rows = []

for _, r in df.iterrows():
    machine = r["machine_clean"]

    if machine not in T120_MACHINES:
        continue

    p = part_level.loc[
        part_level["Child Part"] == r["Child Part"]
    ].iloc[0]

    rows.append({
        "part": r["Child Part"],
        "category": r["Category"],
        "machine": machine,  # normalized
        "daily_demand": p["effective_daily_demand"],
        "inventory": p["Inventory_25"],
        "cycle_time": p["Cycle Time"],
        "net_required_qty": p["net_required_qty"]
    })

dfm = pd.DataFrame(rows)

print("\nDEBUG → dfm shape:", dfm.shape)
print(dfm.head())

if dfm.empty:
    raise ValueError(
        "❌ STILL EMPTY: Check machine names printed above. "
        "They must match MP01, MP05, MP10, MP17"
    )

# =========================================================
# STEP 8: SCORE FUNCTION (HUMAN LOGIC)
# =========================================================
def compute_score(row):
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)

    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    prod_time = qty_if_made * row["cycle_time"]

    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)

dfm["score"] = dfm.apply(compute_score, axis=1)

# =========================================================
# STEP 9: SELECT MAX 3 PARTS PER MACHINE
# =========================================================
selected = []

for machine, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(MAX_PARTS_PER_MACHINE))

selected = pd.concat(selected).reset_index(drop=True)

# =========================================================
# STEP 10: QUANTITY TO PRODUCE (3-DAY INVENTORY LOGIC)
# =========================================================
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]

# =========================================================
# FINAL OUTPUT
# =========================================================
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== FINAL DAILY PLAN (120T MACHINES ONLY) =====")
print(final_plan)


In [ ]:
# ========================= CONFIG =========================
TARGET_COVERAGE_DAYS = 3          # exactly what you want
MAX_PARTS_PER_MACHINE = 3         # your floor rule
MAX_HOURS_PER_DAY = 22
CHANGEOVER_MIN = 40 / 60          # hours

# Filter only non-runners
non_runners = df[df['Category'].isin(['Repeater', 'Stranger'])].copy()

# Net required for next 3 days
non_runners['Net_3day'] = (non_runners['Daily Demand'] * TARGET_COVERAGE_DAYS) - non_runners['Inventory_25']  # use your inventory column
non_runners['Net_3day'] = non_runners['Net_3day'].clip(lower=0)

# Lot size = 3 days worth (big lots = fewer changeovers)
non_runners['Lot_Size'] = (non_runners['Daily Demand'] * TARGET_COVERAGE_DAYS).round(0)
non_runners['Prod_Hours'] = (non_runners['Lot_Size'] * non_runners['Cycle Time']) / 3600

# Urgency score (higher = schedule sooner)
non_runners['Coverage_Days'] = non_runners['Inventory_25'] / non_runners['Daily Demand'].replace(0, 1)
non_runners['Urgency'] = (3 - non_runners['Coverage_Days']) * non_runners['Daily Demand']   # critical first

# Sort: most urgent + same colour together
non_runners = non_runners.sort_values(['Urgency', 'Color'], ascending=[False, True])

# ========================= SCHEDULE =========================
machines = ['MP-01', 'MP-05', 'MP-10', 'MP-11', 'MP-17']   # your 120T machines
schedule = []
machine_load = {m: 0.0 for m in machines}
machine_parts_today = {m: [] for m in machines}

for _, part in non_runners.iterrows():
    if part['Net_3day'] <= 0:
        continue
    
    lot = part['Lot_Size']
    prod_h = part['Prod_Hours']
    color = part['Color']
    
    # Try machines in order of least loaded, prefer same colour
    candidates = sorted(machines, key=lambda m: machine_load[m])
    for m in candidates:
        current_parts = len(machine_parts_today[m])
        if current_parts >= MAX_PARTS_PER_MACHINE:
            continue
        
        setup = CHANGEOVER_MIN if (machine_parts_today[m] and 
                                   machine_parts_today[m][-1]['Color'] != color) else 0
        
        total_h = prod_h + setup
        if machine_load[m] + total_h > MAX_HOURS_PER_DAY:
            continue
        
        # Assign!
        machine_load[m] += total_h
        machine_parts_today[m].append({
            'Part No': part['Part No'],
            'Lot_Size': lot,
            'Prod_Hours': prod_h,
            'Setup_Hours': setup,
            'Color': color,
            'Coverage_After': part['Coverage_Days'] + 3
        })
        
        schedule.append({
            'Machine': m,
            'Part No': part['Part No'],
            'Category': part['Category'],
            'Lot_Size': lot,
            'Total_Hours': total_h,
            'Coverage_Days_After': round(part['Coverage_Days'] + 3, 1)
        })
        break   # assigned, move to next part

# ========================= OUTPUT =========================
print("\n=== 3-DAY INVENTORY PLAN (max 2-3 parts/machine) ===")
for m in machines:
    print(f"\n🛠 {m}  (used {machine_load[m]:.1f}/22 h)")
    if machine_parts_today[m]:
        for p in machine_parts_today[m]:
            print(f"   • {p['Part No']}  ({p['Lot_Size']} pcs)  → {p['Total_Hours']:.1f}h  (setup {p['Setup_Hours']*60:.0f}min)  → {p['Coverage_Days_After']} days stock")
    else:
        print("   (idle or only runners)")

print("\nTotal changeovers today:", sum(1 for m in machine_parts_today.values() if len(m) > 1))

In [ ]:
import pandas as pd
import numpy as np

# ================================================
# CONFIGURATION - Adjust these values as needed
# ================================================

TARGET_COVERAGE_DAYS = 3.0          # Goal: build ~3 days stock when we run a part
MAX_PARTS_PER_MACHINE_PER_DAY = 3   # Your floor rule: max 2-3 parts / machine / day
MAX_HOURS_PER_DAY = 22.0            # Available production hours per machine
CHANGEOVER_HOURS = 40 / 60          # 40 minutes → hours

# Classification thresholds (adjust after seeing your data distribution)
RUNNER_MIN_DEMAND = 500             # pcs/day → daily runners (fixed machines)
REPEATER_MIN_DEMAND = 50            # pcs/day → repeaters (medium frequency)

# Column names in your Excel file - CHANGE THESE TO MATCH YOUR ACTUAL FILE
COL_PART_NO       = "Part No"
COL_DAILY_DEMAND  = "Daily Demand"
COL_CYCLE_TIME    = "Cycle Time"       # in seconds
COL_COLOR         = "Color"
COL_INVENTORY     = "Inventory_25"     # or whatever your inventory column is called
COL_MACHINE       = "Machine Id"       # only used for runners (we ignore for now)

# Your 120T machines (excluding dedicated runner machines if any)
MACHINES = ["MP-01", "MP-05", "MP-10", "MP-11", "MP-17"]

# ================================================
# 1. Load and classify parts
# ================================================

# Replace with your actual file path
FILE_PATH = "D:/Tushar/Capacity_Load_Calculation_Master.xlsx"  # ← CHANGE THIS

df = pd.read_excel(FILE_PATH)

# Basic cleaning
df[COL_DAILY_DEMAND] = pd.to_numeric(df[COL_DAILY_DEMAND], errors='coerce').fillna(0)
df[COL_CYCLE_TIME]   = pd.to_numeric(df[COL_CYCLE_TIME], errors='coerce').fillna(0)
df[COL_INVENTORY]    = pd.to_numeric(df[COL_INVENTORY], errors='coerce').fillna(0)

# Classify Runner / Repeater / Stranger
df['Category'] = 'Stranger'
df.loc[df[COL_DAILY_DEMAND] >= RUNNER_MIN_DEMAND, 'Category'] = 'Runner'
df.loc[(df[COL_DAILY_DEMAND] >= REPEATER_MIN_DEMAND) & 
       (df[COL_DAILY_DEMAND] < RUNNER_MIN_DEMAND), 'Category'] = 'Repeater'

print("Part classification summary:")
print(df['Category'].value_counts())
print("\nTop 10 highest demand parts:")
print(df[[COL_PART_NO, COL_DAILY_DEMAND, 'Category', COL_COLOR]].sort_values(COL_DAILY_DEMAND, ascending=False).head(10))

# We only schedule Repeaters + Strangers
non_runners = df[df['Category'].isin(['Repeater', 'Stranger'])].copy()

if non_runners.empty:
    print("No Repeater or Stranger parts found. Nothing to schedule.")
    exit()

# ================================================
# 2. Prepare planning data
# ================================================

# Net requirement for next TARGET_COVERAGE_DAYS
non_runners['Net_Required'] = (non_runners[COL_DAILY_DEMAND] * TARGET_COVERAGE_DAYS) - non_runners[COL_INVENTORY]
non_runners['Net_Required'] = non_runners['Net_Required'].clip(lower=0)

# Lot size = target coverage amount (big lots)
non_runners['Lot_Size'] = (non_runners[COL_DAILY_DEMAND] * TARGET_COVERAGE_DAYS).round(0).astype(int)

# Production hours for the lot
non_runners['Prod_Hours'] = (non_runners['Lot_Size'] * non_runners[COL_CYCLE_TIME]) / 3600

# Current days of coverage
non_runners['Current_Coverage_Days'] = non_runners[COL_INVENTORY] / non_runners[COL_DAILY_DEMAND].replace(0, 0.001)

# Urgency: how badly we need this part (higher = more urgent)
non_runners['Urgency'] = (TARGET_COVERAGE_DAYS - non_runners['Current_Coverage_Days'].clip(lower=0)) * non_runners[COL_DAILY_DEMAND]

# Sort: most urgent first, then same color to minimize changeovers
non_runners = non_runners.sort_values(['Urgency', COL_COLOR], ascending=[False, True]).reset_index(drop=True)

# ================================================
# 3. Schedule - greedy assignment
# ================================================

schedule = []
machine_usage = {m: 0.0 for m in MACHINES}
machine_parts_today = {m: [] for m in MACHINES}

for idx, part in non_runners.iterrows():
    if part['Net_Required'] <= 0:
        continue

    lot_qty = part['Lot_Size']
    prod_h = part['Prod_Hours']
    color = part.get(COL_COLOR, 'Unknown')

    # Try machines - prefer least loaded, then same color if possible
    for machine in sorted(MACHINES, key=lambda m: machine_usage[m]):
        parts_today = machine_parts_today[machine]
        if len(parts_today) >= MAX_PARTS_PER_MACHINE_PER_DAY:
            continue

        # Setup time
        if parts_today:
            last_color = parts_today[-1].get('Color', 'Unknown')
            setup_h = CHANGEOVER_HOURS if last_color != color else 0.0
        else:
            setup_h = CHANGEOVER_HOURS  # first part always has setup

        total_h = prod_h + setup_h

        # Can we fit?
        if machine_usage[machine] + total_h <= MAX_HOURS_PER_DAY:
            # Assign
            machine_usage[machine] += total_h
            entry = {
                'Part_No': part[COL_PART_NO],
                'Category': part['Category'],
                'Lot_Size': lot_qty,
                'Prod_Hours': round(prod_h, 2),
                'Setup_Hours': round(setup_h, 2),
                'Total_Hours': round(total_h, 2),
                'Color': color,
                'Coverage_After': round(part['Current_Coverage_Days'] + TARGET_COVERAGE_DAYS, 1)
            }
            parts_today.append(entry)
            schedule.append({**entry, 'Machine': machine})
            break  # assigned → next part

# ================================================
# 4. Print nice schedule
# ================================================

print("\n" + "="*80)
print("          3-DAY INVENTORY BUILD PLAN (Repeater + Stranger only)")
print("          Max 2-3 parts per machine • Focus on big lots • Minimal changeovers")
print("="*80)

total_changeovers = 0
total_hours_used = 0

for machine in MACHINES:
    parts = machine_parts_today.get(machine, [])
    hours = machine_usage.get(machine, 0.0)
    total_hours_used += hours
    
    print(f"\n🛠 {machine:6}   used {hours:5.1f} / {MAX_HOURS_PER_DAY:.1f} h   ({hours/MAX_HOURS_PER_DAY*100:5.1f}%)")
    
    if not parts:
        print("   (no parts scheduled today - runners only or idle)")
        continue
    
    changeovers = sum(1 for i in range(1, len(parts)) if parts[i]['Color'] != parts[i-1]['Color'])
    total_changeovers += changeovers
    
    for p in parts:
        setup_min = p['Setup_Hours'] * 60
        print(f"   • {p['Part_No']:18}  {p['Lot_Size']:>6} pcs   "
              f"{p['Prod_Hours']:>5.1f}h prod + {setup_min:>3.0f}min setup   "
              f"→ {p['Total_Hours']:>5.1f}h   → {p['Coverage_After']:>5.1f} days stock")

print("\n" + "-"*80)
print(f"Summary:")
print(f"  • Total machines used: {len([m for m in machine_usage.values() if m > 0])} / {len(MACHINES)}")
print(f"  • Total production hours: {total_hours_used:.1f} h")
print(f"  • Estimated changeovers today: {total_changeovers}")
print(f"  • Average parts per active machine: {len(schedule)/len([m for m in machine_usage.values() if m > 0]):.1f}")
print(f"  • Scheduled lots: {len(schedule)}")
print("-"*80)

# Optional: save schedule
pd.DataFrame(schedule).to_excel("today_3day_plan.xlsx", index=False)
print("Schedule saved → today_3day_plan.xlsx")

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

# ================================================
# CONFIG - Tweak these based on your floor reality
# ================================================
FILE_PATH = "Copy of Master Data_290102026 2 - Copy.xlsx"   # ← your exact file name

TARGET_COVERAGE_DAYS = 3.0          # Build up to 3 days buffer when we run a part
MAX_PARTS_PER_MACHINE = 3           # Your strict rule: max 2-3 parts / machine / day
MAX_HOURS_PER_DAY = 22.0
CHANGEOVER_MIN = 40                 # minutes
CHANGEOVER_HOURS = CHANGEOVER_MIN / 60

# RRS thresholds (adjust after first run)
RUNNER_DEMAND_THRESHOLD = 500       # Runner = daily high-volume (fixed machines)
REPEATER_DEMAND_THRESHOLD = 50      # Repeater = medium, we build 3-day lots

# Sheet name (from your previous notebooks)
SHEET_NAME = "Master Data "

# ================================================
# 1. LOAD & AGGREGATE CORRECTLY (exactly as you described)
# ================================================
master = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

# Clean columns
master["Daily Plan"] = pd.to_numeric(master["Daily Plan"], errors="coerce").fillna(0)
master["Sub Count"] = pd.to_numeric(master["Sub Count"], errors="coerce").fillna(0)
master["Inventory_25"] = pd.to_numeric(master["Inventory_25"], errors="coerce").fillna(0)
master[" Minimum Quantity"] = pd.to_numeric(master[" Minimum Quantity"], errors="coerce").fillna(0)
master["Cycle Time"] = pd.to_numeric(master["Cycle Time"], errors="coerce").fillna(0)   # seconds assumed

# Aggregate per unique Child Part
agg = []
for child, group in master.groupby("Child Part"):
    daily_needed = (group["Daily Plan"] * group["Sub Count"]).sum()
    min_qty = group[" Minimum Quantity"].iloc[0]          # add only once
    inventory = group["Inventory_25"].iloc[0]
    cycle_time = group["Cycle Time"].iloc[0]              # assume same for the part
    color = group["Color"].iloc[0] if "Color" in group.columns else "Unknown"
    machines_raw = ",".join(group["Vertical Machines"].astype(str).unique())
    
    net_required = daily_needed + min_qty - inventory
    net_required = max(0, net_required)
    
    agg.append({
        "Child Part": child,
        "Daily_Demand": daily_needed,          # today's true demand
        "Net_Required": net_required,          # daily + one-time buffer - inv
        "Cycle_Time_sec": cycle_time,
        "Color": color,
        "Vertical_Machines": machines_raw,
        "Current_Inventory": inventory
    })

df = pd.DataFrame(agg)

# Classify Runner / Repeater / Stranger
df["Category"] = "Stranger"
df.loc[df["Daily_Demand"] >= RUNNER_DEMAND_THRESHOLD, "Category"] = "Runner"
df.loc[(df["Daily_Demand"] >= REPEATER_DEMAND_THRESHOLD) & 
       (df["Daily_Demand"] < RUNNER_DEMAND_THRESHOLD), "Category"] = "Repeater"

print("Classification:")
print(df["Category"].value_counts())
print("\nSample (highest net required):")
print(df.sort_values("Net_Required", ascending=False).head(10)[["Child Part","Daily_Demand","Net_Required","Category"]])

# We only schedule Repeater + Stranger (Runners are on fixed machines)
to_schedule = df[df["Category"].isin(["Repeater", "Stranger"])].copy()

# ================================================
# 2. PREPARE ELIGIBLE MACHINES PER PART
# ================================================
def normalize_machines(s):
    if pd.isna(s): return []
    return [m.strip().upper() for m in str(s).split(",") if m.strip()]

to_schedule["Eligible_Machines"] = to_schedule["Vertical_Machines"].apply(normalize_machines)

# ================================================
# 3. TWO-PHASE SCHEDULER (Daily first → Buffer second)
# ================================================
machines = ["MP-01", "MP-05", "MP-10", "MP-11", "MP-17"]
machine_load = {m: 0.0 for m in machines}
machine_sequence = {m: [] for m in machines}   # list of parts in order (for color grouping)

schedule = []

# PHASE 1: Guarantee TODAY'S DAILY DEMAND (split if needed)
print("\n=== PHASE 1: Covering today's daily demand ===")
daily_df = to_schedule.copy()
daily_df["Remaining_Daily"] = daily_df["Daily_Demand"]

for _, part in daily_df.iterrows():
    if part["Remaining_Daily"] <= 0:
        continue
    
    prod_time_per_piece = part["Cycle_Time_sec"] / 3600
    qty_left = part["Remaining_Daily"]
    eligible = part["Eligible_Machines"]
    
    for m in eligible:
        if m not in machines: continue
        free = MAX_HOURS_PER_DAY - machine_load[m]
        if free <= CHANGEOVER_HOURS: continue
            
        # Setup only if first part or different color
        setup_h = CHANGEOVER_HOURS if not machine_sequence[m] or machine_sequence[m][-1]["Color"] != part["Color"] else 0
        free -= setup_h
        if free <= 0: continue
        
        max_qty_fit = free / prod_time_per_piece
        assign_qty = min(qty_left, max_qty_fit)
        if assign_qty <= 0: continue
        
        assign_hours = assign_qty * prod_time_per_piece
        
        # Record
        machine_load[m] += assign_hours + setup_h
        machine_sequence[m].append({"Child Part": part["Child Part"], "Color": part["Color"], "Qty": assign_qty, "Hours": assign_hours, "Setup": setup_h})
        
        schedule.append({
            "Machine": m,
            "Child Part": part["Child Part"],
            "Qty": round(assign_qty, 0),
            "Type": "Daily",
            "Hours": round(assign_hours + setup_h, 2),
            "Color": part["Color"]
        })
        
        qty_left -= assign_qty
        if qty_left <= 0: break

# PHASE 2: Add buffer (big lots) to Repeaters + urgent Strangers, max 3 parts/machine total
print("\n=== PHASE 2: Adding 3-day buffer where capacity allows ===")
buffer_df = to_schedule[to_schedule["Net_Required"] > to_schedule["Daily_Demand"]].copy()  # only parts that still need buffer
buffer_df["Buffer_Qty"] = buffer_df["Net_Required"] - buffer_df["Daily_Demand"]
buffer_df = buffer_df.sort_values("Buffer_Qty", ascending=False)

for _, part in buffer_df.iterrows():
    qty_left = part["Buffer_Qty"]
    if qty_left <= 0: continue
    
    prod_time_per_piece = part["Cycle_Time_sec"] / 3600
    eligible = part["Eligible_Machines"]
    lot_target = part["Daily_Demand"] * TARGET_COVERAGE_DAYS   # big lot
    
    # Try to assign big lot (or portion) without exceeding 3 parts/machine
    for m in sorted(eligible, key=lambda x: machine_load.get(x, 0)):
        if m not in machines: continue
        if len(machine_sequence[m]) >= MAX_PARTS_PER_MACHINE: continue
            
        free = MAX_HOURS_PER_DAY - machine_load[m]
        if free <= CHANGEOVER_HOURS: continue
            
        setup_h = CHANGEOVER_HOURS if not machine_sequence[m] or machine_sequence[m][-1]["Color"] != part["Color"] else 0
        free -= setup_h
        if free <= 0: continue
        
        max_qty_fit = free / prod_time_per_piece
        assign_qty = min(qty_left, lot_target, max_qty_fit)
        if assign_qty < 50: continue   # ignore tiny buffer adds
        
        assign_hours = assign_qty * prod_time_per_piece
        
        machine_load[m] += assign_hours + setup_h
        machine_sequence[m].append({"Child Part": part["Child Part"], "Color": part["Color"], "Qty": assign_qty, "Hours": assign_hours, "Setup": setup_h})
        
        schedule.append({
            "Machine": m,
            "Child Part": part["Child Part"],
            "Qty": round(assign_qty, 0),
            "Type": "Buffer",
            "Hours": round(assign_hours + setup_h, 2),
            "Color": part["Color"]
        })
        
        qty_left -= assign_qty
        if qty_left <= 0: break

# ================================================
# 4. FINAL OUTPUT
# ================================================
print("\n" + "="*90)
print("FINAL DAILY PLAN (Daily demand guaranteed + smart buffer)")
print("="*90)

total_hours = 0
total_changeovers = 0

for m in machines:
    parts = machine_sequence.get(m, [])
    hours = machine_load.get(m, 0)
    total_hours += hours
    
    print(f"\n🛠 {m}   {hours:6.1f} / {MAX_HOURS_PER_DAY:.1f} h   ({hours/MAX_HOURS_PER_DAY*100:5.1f}%)")
    if not parts:
        print("   No Repeater/Stranger assigned")
        continue
    
    changeovers = sum(1 for i in range(1, len(parts)) if parts[i]["Color"] != parts[i-1]["Color"])
    total_changeovers += changeovers
    
    for p in parts:
        setup_min = p["Setup"] * 60
        print(f"   {p['Child Part']:20}  {p['Qty']:>6} pcs   {p['Hours']:>5.1f}h  (setup {setup_min:>3.0f} min)   {p['Type']}")

print("\n" + "-"*90)
print(f"Total hours used     : {total_hours:.1f} h across all machines")
print(f"Estimated changeovers: {total_changeovers}")
print(f"Parts scheduled      : {len(schedule)}")
print(f"Daily demand covered : 100% (by design)")
print("-"*90)

# Save
pd.DataFrame(schedule).to_excel("daily_plan_with_buffer.xlsx", index=False)
print("Saved → daily_plan_with_buffer.xlsx")

In [ ]:
import pandas as pd
import numpy as np

# ================================================
# CONFIGURATION
# ================================================
FILE_PATH = "Copy of Master Data_290102026 2 - Copy.xlsx"

TARGET_COVERAGE_DAYS = 3.0
MAX_PARTS_PER_MACHINE = 3           # max 2-3 parts per machine per day
MAX_HOURS_PER_DAY = 22.0
CHANGEOVER_MIN = 40
CHANGEOVER_HOURS = CHANGEOVER_MIN / 60.0

# Classification thresholds (you can tune after seeing counts)
RUNNER_THRESHOLD = 2000             # daily demand ≥ 2000 → Runner
REPEATER_THRESHOLD = 200            # 200 ≤ daily demand < 2000 → Repeater

# Allowed machines (expanded based on your screenshot)
ALLOWED_MACHINES = [
    "MP-01", "MP-04", "MP-05", "MP-08", "MP-10", "MP-11", "MP-17",
    "TOYO-IST", "TOYO1ST", "TOYO IST"
]

SHEET_NAME = "Master Data "

# ================================================
# MACHINE NAME NORMALIZATION
# ================================================
def clean_machine(m):
    if pd.isna(m) or not str(m).strip():
        return None
    s = str(m).strip().upper()
    s = s.replace(".", "-")
    s = s.replace("M.P-", "MP-")
    s = s.replace("MP.", "MP-")
    s = s.replace("TOYOI ST", "TOYO-IST")
    s = s.replace("TOYO IST", "TOYO-IST")
    s = s.replace("TOYO1ST", "TOYO-IST")
    s = s.replace(" ", "-")
    return s

def parse_machines(cell):
    if pd.isna(cell):
        return []
    raw = str(cell).split(",")
    cleaned = [clean_machine(x) for x in raw if clean_machine(x)]
    return list(set(cleaned))  # remove duplicates

# ================================================
# 1. LOAD DATA & AGGREGATE PER CHILD PART
# ================================================
print("Loading Excel file...")
master = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

# Ensure numeric columns
numeric_cols = ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity", "Cycle Time"]
for col in numeric_cols:
    master[col] = pd.to_numeric(master[col], errors="coerce").fillna(0)

# Aggregate
agg = []
for child, group in master.groupby("Child Part"):
    daily_demand = (group["Daily Plan"] * group["Sub Count"]).sum()
    min_qty = group["Minimum Quantity"].iloc[0]          # once only
    inventory = group["Inventory_25"].iloc[0]
    cycle_time_sec = group["Cycle Time"].iloc[0]
    vm_raw = group["Vertical Machines"].dropna().unique()
    vm_str = ",".join(vm_raw.astype(str))

    net_required = daily_demand + min_qty - inventory
    net_required = max(0, net_required)

    agg.append({
        "Child Part": child,
        "Daily_Demand": daily_demand,
        "Net_Required": net_required,
        "Cycle_Time_sec": cycle_time_sec,
        "Vertical_Machines_raw": vm_str,
        "Current_Inventory": inventory
    })

df = pd.DataFrame(agg)

# Classify Runner / Repeater / Stranger
df["Category"] = "Stranger"
df.loc[df["Daily_Demand"] >= RUNNER_THRESHOLD, "Category"] = "Runner"
df.loc[(df["Daily_Demand"] >= REPEATER_THRESHOLD) & 
       (df["Daily_Demand"] < RUNNER_THRESHOLD), "Category"] = "Repeater"

print("\nClassification counts:")
print(df["Category"].value_counts())
print("\nTop 10 highest daily demand:")
print(df.sort_values("Daily_Demand", ascending=False).head(10)[
    ["Child Part", "Daily_Demand", "Net_Required", "Category"]
])

# Only schedule Repeater + Stranger
to_schedule = df[df["Category"].isin(["Repeater", "Stranger"])].copy()
to_schedule["Eligible_Machines"] = to_schedule["Vertical_Machines_raw"].apply(parse_machines)

print("\nSample parts with parsed machines:")
print(to_schedule[["Child Part", "Daily_Demand", "Eligible_Machines", "Category"]].head(12))

# ================================================
# 2. SCHEDULING SETUP
# ================================================
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_sequence = {m: [] for m in ALLOWED_MACHINES}
schedule_records = []

# PHASE 1: Cover TODAY's demand first
print("\n=== PHASE 1 – Covering today's demand ===")
daily_parts = to_schedule[to_schedule["Daily_Demand"] > 0].copy()
daily_parts["Remaining"] = daily_parts["Daily_Demand"]

for _, row in daily_parts.iterrows():
    if row["Remaining"] <= 0:
        continue

    sec_per_pc = row["Cycle_Time_sec"]
    if sec_per_pc <= 0:
        continue

    hrs_per_pc = sec_per_pc / 3600.0
    qty_left = row["Remaining"]
    eligible = [m for m in row["Eligible_Machines"] if m in ALLOWED_MACHINES]

    if not eligible:
        continue

    for m in sorted(eligible, key=lambda x: machine_load[x]):
        free_h = MAX_HOURS_PER_DAY - machine_load[m]
        if free_h <= CHANGEOVER_HOURS:
            continue

        last_color = machine_sequence[m][-1].get("Color", None) if machine_sequence[m] else None
        setup_h = CHANGEOVER_HOURS if last_color != row.get("Color") else 0.0   # Color not present, so usually full setup

        free_after_setup = free_h - setup_h
        if free_after_setup <= 0:
            continue

        max_qty = free_after_setup / hrs_per_pc
        assign_qty = min(qty_left, max_qty)
        if assign_qty < 5:
            continue

        assign_h = assign_qty * hrs_per_pc

        machine_load[m] += assign_h + setup_h
        machine_sequence[m].append({
            "Child Part": row["Child Part"],
            "Qty": assign_qty,
            "Hours": assign_h,
            "Setup_h": setup_h,
            "Type": "Daily"
        })

        schedule_records.append({
            "Machine": m,
            "Child Part": row["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Type": "Daily"
        })

        qty_left -= assign_qty
        row["Remaining"] = qty_left
        if qty_left <= 0:
            break

# PHASE 2: Build buffer (3-day lots)
print("\n=== PHASE 2 – Adding 3-day buffer ===")
buffer_parts = to_schedule[to_schedule["Net_Required"] > to_schedule["Daily_Demand"]].copy()
buffer_parts["Buffer_Qty"] = buffer_parts["Net_Required"] - buffer_parts["Daily_Demand"]
buffer_parts = buffer_parts.sort_values("Buffer_Qty", ascending=False)

for _, row in buffer_parts.iterrows():
    qty_left = row["Buffer_Qty"]
    if qty_left <= 0:
        continue

    sec_per_pc = row["Cycle_Time_sec"]
    if sec_per_pc <= 0:
        continue

    hrs_per_pc = sec_per_pc / 3600.0
    eligible = [m for m in row["Eligible_Machines"] if m in ALLOWED_MACHINES]
    if not eligible:
        continue

    lot_target = row["Daily_Demand"] * TARGET_COVERAGE_DAYS

    for m in sorted(eligible, key=lambda x: machine_load[x]):
        if len(machine_sequence[m]) >= MAX_PARTS_PER_MACHINE:
            continue

        free_h = MAX_HOURS_PER_DAY - machine_load[m]
        if free_h <= CHANGEOVER_HOURS:
            continue

        last_color = machine_sequence[m][-1].get("Color", None) if machine_sequence[m] else None
        setup_h = CHANGEOVER_HOURS if last_color != row.get("Color") else 0.0

        free_after_setup = free_h - setup_h
        if free_after_setup <= 0:
            continue

        max_qty = free_after_setup / hrs_per_pc
        assign_qty = min(qty_left, lot_target, max_qty)
        if assign_qty < 10:
            continue

        assign_h = assign_qty * hrs_per_pc

        machine_load[m] += assign_h + setup_h
        machine_sequence[m].append({
            "Child Part": row["Child Part"],
            "Qty": assign_qty,
            "Hours": assign_h,
            "Setup_h": setup_h,
            "Type": "Buffer"
        })

        schedule_records.append({
            "Machine": m,
            "Child Part": row["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Type": "Buffer"
        })

        qty_left -= assign_qty
        if qty_left <= 0:
            break

# ================================================
# FINAL OUTPUT
# ================================================
print("\n" + "="*100)
print("FINAL DAILY PRODUCTION PLAN – 120T MACHINES")
print("="*100)

total_h = sum(machine_load.values())
total_chg = 0
active_machines = 0

for m in sorted(ALLOWED_MACHINES):
    seq = machine_sequence.get(m, [])
    h = machine_load.get(m, 0.0)
    if h == 0 and not seq:
        continue

    active_machines += 1
    chg = sum(1 for i in range(1, len(seq)) if seq[i].get("Color") != seq[i-1].get("Color"))
    total_chg += chg

    print(f"\n🛠 {m:8}   {h:6.1f} / {MAX_HOURS_PER_DAY:.1f} h   ({h/MAX_HOURS_PER_DAY*100:5.1f}%)   "
          f"{len(seq)} parts   {chg} changeovers")

    for item in seq:
        print(f"   • {item['Child Part']:22} {item['Qty']:>7,.0f} pcs   "
              f"{item['Hours']:>5.1f}h   setup {item['Setup_h']*60:>3.0f} min   {item['Type']}")

print("\n" + "-"*100)
print(f"Total hours used       : {total_h:.1f} h")
print(f"Total changeovers      : {total_chg}")
print(f"Parts scheduled        : {len(schedule_records)}")
print(f"Active machines        : {active_machines} / {len(ALLOWED_MACHINES)}")
print(f"Average utilization    : {total_h / (len(ALLOWED_MACHINES) * MAX_HOURS_PER_DAY) * 100:.1f}%")
print("-"*100)

# Save
if schedule_records:
    pd.DataFrame(schedule_records).to_excel("daily_plan_120t_fixed.xlsx", index=False)
    print("Plan saved → daily_plan_120t_fixed.xlsx")
else:
    print("No parts were scheduled. Check:")
    print("  • Machine name matching")
    print("  • Cycle time values (are they seconds or minutes?)")
    print("  • Demand values for Repeater/Stranger parts")